B4 executing following code first go to runtime->change rutime execution -> T4 GPU


In [ ]:
!nvidia-smi

In [ ]:
!apt-get install -y nvidia-cuda-toolkit

In [ ]:
%%writefile vector_add.cu

#include <iostream>
#include <chrono>
#include <cstdlib> // Required for rand()

using namespace std;
using namespace chrono;

// CUDA Kernel
__global__ void add(int *a, int *b, int *c, int n) {

    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n) {
        c[i] = a[i] + b[i];
    }
}

int main() {

    int n;

    cout << "Enter size of vectors: ";
    cin >> n;

    int *a = new int[n];
    int *b = new int[n];
    int *c = new int[n];

    // ---------------- RANDOM ARRAY GENERATION ----------------
    // Generates random numbers between 0 and 999
    for (int i = 0; i < n; i++) {
        a[i] = rand() % 1000;
        b[i] = rand() % 1000;
    }
    
    cout << "Successfully generated " << n << " random elements for Vectors A and B.\n";

    // ---------------- SEQUENTIAL ADDITION ----------------

    auto start = high_resolution_clock::now();

    for (int i = 0; i < n; i++) {
        c[i] = a[i] + b[i];
    }

    auto stop = high_resolution_clock::now();

    auto duration = duration_cast<microseconds>(stop - start);

    cout << "\nSequential Time: "
         << duration.count() << " microseconds\n";

    // ---------------- CUDA PART ----------------

    int *d_a, *d_b, *d_c;

    cudaMalloc(&d_a, n * sizeof(int));
    cudaMalloc(&d_b, n * sizeof(int));
    cudaMalloc(&d_c, n * sizeof(int));

    cudaMemcpy(d_a, a, n * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, n * sizeof(int), cudaMemcpyHostToDevice);

    int threads = 256;
    int blocks = (n + threads - 1) / threads;

    start = high_resolution_clock::now();

    add<<<blocks, threads>>>(d_a, d_b, d_c, n);

    cudaDeviceSynchronize();

    stop = high_resolution_clock::now();

    duration = duration_cast<microseconds>(stop - start);

    cout << "Parallel CUDA Time: "
         << duration.count() << " microseconds\n";

    cudaMemcpy(c, d_c, n * sizeof(int), cudaMemcpyDeviceToHost);

    // ---------------- SAFE PRINTING ----------------
    // Only print the first 10 elements to prevent the notebook from freezing
    cout << "\nResult Vector (first 10 elements): ";
    
    int print_limit = (n < 10) ? n : 10;
    
    for (int i = 0; i < print_limit; i++) {
        cout << c[i] << " ";
    }
    
    cout << "...\n";

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    delete[] a;
    delete[] b;
    delete[] c;

    return 0;
}

In [ ]:
!nvcc vector_add.cu -o vector_add

In [ ]:
!./vector_add

In [ ]:
%%writefile vector_mul.cu

#include <iostream>
#include <chrono>
#include <cstdlib>

using namespace std;
using namespace chrono;

// CUDA Kernel for Multiplication
__global__ void multiply_vectors(int *a, int *b, int *c, int n) {

    // Calculate global thread ID
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // Make sure we don't go out of bounds
    if (i < n) {
        c[i] = a[i] * b[i]; // <--- Changed from addition to multiplication
    }
}

int main() {

    int n;

    cout << "Enter size of vectors: ";
    cin >> n;

    int *a = new int[n];
    int *b = new int[n];
    int *c = new int[n];

    // ---------------- RANDOM ARRAY GENERATION ----------------
    // Generates random numbers between 0 and 999
    for (int i = 0; i < n; i++) {
        a[i] = rand() % 1000;
        b[i] = rand() % 1000;
    }
    
    cout << "Successfully generated " << n << " random elements for Vectors A and B.\n";

    // ---------------- SEQUENTIAL MULTIPLICATION ----------------

    auto start = high_resolution_clock::now();

    for (int i = 0; i < n; i++) {
        c[i] = a[i] * b[i];
    }

    auto stop = high_resolution_clock::now();
    auto duration = duration_cast<microseconds>(stop - start);

    cout << "\nSequential Time: "
         << duration.count() << " microseconds\n";

    // ---------------- CUDA PART ----------------

    int *d_a, *d_b, *d_c;

    // Allocate memory on the GPU
    cudaMalloc(&d_a, n * sizeof(int));
    cudaMalloc(&d_b, n * sizeof(int));
    cudaMalloc(&d_c, n * sizeof(int));

    // Copy data from CPU RAM to GPU VRAM
    cudaMemcpy(d_a, a, n * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, n * sizeof(int), cudaMemcpyHostToDevice);

    // Define grid and block dimensions
    int threads = 256;
    int blocks = (n + threads - 1) / threads;

    // Start timing the GPU execution
    start = high_resolution_clock::now();

    // Launch the CUDA Kernel
    multiply_vectors<<<blocks, threads>>>(d_a, d_b, d_c, n);

    // Wait for GPU to finish before stopping the timer
    cudaDeviceSynchronize();

    stop = high_resolution_clock::now();
    duration = duration_cast<microseconds>(stop - start);

    cout << "Parallel CUDA Time: "
         << duration.count() << " microseconds\n";

    // Copy the result back from GPU VRAM to CPU RAM
    cudaMemcpy(c, d_c, n * sizeof(int), cudaMemcpyDeviceToHost);

    // ---------------- SAFE PRINTING ----------------
    cout << "\nResult Vector (first 10 elements): ";
    
    int print_limit = (n < 10) ? n : 10;
    
    for (int i = 0; i < print_limit; i++) {
        cout << c[i] << " ";
    }
    
    cout << "...\n";

    // Free memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    delete[] a;
    delete[] b;
    delete[] c;

    return 0;
}

In [ ]:
!nvcc vector_mul.cu.cu -o matrix_mul

In [ ]:
!./matrix_mul